# 1. Data acquisition and provenance

How this project's headline scale claim -- **164,209 provenance-tracked
source records** -- is built up from real archive retrievals, and what a
single manifest entry actually records.


This notebook is part of the reproducibility set for **Finding Earth 2.0 in
Distant Worlds**. It reads the same committed data every other output in this
project reads (`results/`, `data/processed/`, `data/manifests/`) and calls
the same `earth2` functions the pipeline itself calls -- nothing here is a
simplified restatement computed a different way. Run `python -m earth2 all`
first if `results/` does not exist yet.

See `docs/METHODS.md` for the full equations and `docs/LIMITATIONS.md` for
this project's stated caveats.


In [1]:
import json
import sys
sys.path.insert(0, "../src")

import pandas as pd

from earth2.config import ROOT
from earth2.provenance import ManifestStore

store = ManifestStore()
manifests = store.all()
print(f"{len(manifests)} manifest files in data/manifests/")
print(f"total_source_records() = {store.total_source_records():,}")


13 manifest files in data/manifests/
total_source_records() = 164,209


## Every retrieval, individually

Each row below is one archive retrieval: the literal query it ran, when, and
a SHA-256 of the payload as received. This -- not a hand-typed table -- is
where the `164,209` figure comes from.


In [2]:
rows = store.summary_rows()
df = pd.DataFrame(rows).sort_values("n_rows", ascending=False)
df[["dataset_id", "archive", "source_table", "n_rows", "status", "sha256_short"]]


,dataset_id,archive,source_table,n_rows,status,sha256_short
9,nasa_stellarhosts,NASA Exoplanet Archive,stellarhosts,47857,ok,45513b70c576
6,nasa_ps,NASA Exoplanet Archive,ps,40106,ok,5ea9c394849d
10,nasa_tce_dr25,NASA Exoplanet Archive,q1_q17_dr25_tce,34032,ok,246512d90dc8
11,nasa_toi,NASA Exoplanet Archive,toi,8136,ok,77b693084fe4
4,nasa_koi_dr25,NASA Exoplanet Archive,q1_q17_dr25_koi,8054,ok,5a62619a39df
7,nasa_pscomppars,NASA Exoplanet Archive,pscomppars,6354,ok,73eed33e0d57
12,nasa_transitspec,NASA Exoplanet Archive,transitspec,5948,ok,ace18f6adaee
0,gaia_dr3_crossmatch,Gaia Archive (ESA),gaiadr3.gaia_source,4408,ok,d7c5fcfe56f4
3,nasa_k2pandc,NASA Exoplanet Archive,k2pandc,4068,ok,1444b1fa2b15
2,nasa_emissionspec,NASA Exoplanet Archive,emissionspec,2361,ok,8d78a5057343


In [3]:
assert df["n_rows"].sum() == store.total_source_records()
print(f"Sum of every retrieval's n_rows: {df['n_rows'].sum():,} "
      f"(matches total_source_records(): {store.total_source_records():,})")


Sum of every retrieval's n_rows: 164,209 (matches total_source_records(): 164,209)


## What a manifest actually contains

The literal ADQL/API query, the resolved URL, the retrieval timestamp, the
row/column counts, and the payload hash -- everything needed to either
re-issue the same query or detect that the archive's answer has since
changed (see `docs/DATA_SOURCES.md`, "Raw payload policy").


In [4]:
spine = next(m for m in manifests if m.dataset_id == "nasa_pscomppars")
print(json.dumps(spine.to_dict(), indent=2)[:1200])


{
  "dataset_id": "nasa_pscomppars",
  "archive": "NASA Exoplanet Archive",
  "source_table": "pscomppars",
  "query": "select pl_name, hostname, pl_letter, sy_snum, sy_pnum, cb_flag, discoverymethod, disc_year, disc_facility, pl_orbper, pl_orbpererr1, pl_orbpererr2, pl_orbperlim, pl_orbsmax, pl_orbsmaxerr1, pl_orbsmaxerr2, pl_orbsmaxlim, pl_rade, pl_radeerr1, pl_radeerr2, pl_radelim, pl_bmasse, pl_bmasseerr1, pl_bmasseerr2, pl_bmasselim, pl_dens, pl_denserr1, pl_denserr2, pl_denslim, pl_orbeccen, pl_orbeccenerr1, pl_orbeccenerr2, pl_orbeccenlim, pl_insol, pl_insolerr1, pl_insolerr2, pl_insollim, pl_eqt, pl_eqterr1, pl_eqterr2, pl_eqtlim, pl_orbincl, pl_orbinclerr1, pl_orbinclerr2, pl_orbincllim, pl_trandep, pl_trandeperr1, pl_trandeperr2, pl_trandeplim, pl_trandur, pl_trandurerr1, pl_trandurerr2, pl_trandurlim, pl_ratdor, pl_ratdorerr1, pl_ratdorerr2, pl_ratdorlim, pl_ratror, pl_ratrorerr1, pl_ratrorerr2, pl_ratrorlim, pl_imppar, pl_impparerr1, pl_impparerr2, pl_impparlim, pl_tranmid,

## Why 164,209 is a row count, not a planet count

`pscomppars` (the analysis spine, one row per confirmed planet) is only one
of twelve NASA tables retrieved, plus the Gaia DR3 crossmatch. `ps` alone
contributes 40,106 *per-publication* rows for the same 6,354 planets. Adding
every table's row count together is a defensible **data-scale** claim
precisely because it is never presented as a planet, spectrum, or
observation count -- see the wording discipline in `docs/DATA_SOURCES.md`.


In [5]:
scale = json.loads((ROOT / "results" / "analysis_summary.json").read_text())["scale"]
pop = json.loads((ROOT / "results" / "analysis_summary.json").read_text())["population"]
print(f"Source records ingested:  {scale['total_source_records']:,}")
print(f"Confirmed planets analysed: {pop['n_confirmed_planets']:,}")
print(f"Unique host systems:        {pop['n_unique_host_systems']:,}")


Source records ingested:  164,209
Confirmed planets analysed: 6,354
Unique host systems:        4,764
